## Setup

### Import Libraries

In [67]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from tqdm import tqdm

from torchvision import datasets
from torchvision import transforms

from pathlib import Path

import owncloud
from torch.utils.data import random_split

### Utility Functions

In [59]:
def create_model(n_features, n_classes, hidden_layer_size: int = 64, SEED=2026) -> nn.Sequential:
    # create model
    torch.manual_seed(SEED)

    model = nn.Sequential(
        nn.Linear(n_features, hidden_layer_size),
        nn.ReLU(),
        nn.Linear(hidden_layer_size, n_classes),
    )
    return model

def create_deep_model(n_features, n_classes, hidden_layer_sizes: list[int] = [1024, 512, 256, 128, 64, 32], SEED=2026) -> nn.Sequential:
    # create model
    torch.manual_seed(SEED)

    layers = []
    input_size = n_features

    for hidden_layer_size in hidden_layer_sizes:
        layers.append(nn.Linear(input_size, hidden_layer_size))
        layers.append(nn.ReLU())
        input_size = hidden_layer_size

    layers.append(nn.Linear(input_size, n_classes))

    model = nn.Sequential(*layers)
    return model

def train(model: nn.Module, features_train: torch.Tensor, labels_train: torch.Tensor, optimizer, loss_function = nn.CrossEntropyLoss()) -> None:

    optimizer.zero_grad()

    output = model.forward(features_train)

    loss = loss_function(output, labels_train)

    loss.backward()

    optimizer.step()

def calculate_accuracy(model: nn.Module, features: torch.Tensor, labels: torch.Tensor, verbose = True) -> torch.Tensor: 

    output = model(features)
    accuracy = (labels == torch.argmax(output, dim=1)).sum()/len(labels)
    if verbose:
        print(f'Accuracy: {(accuracy*100).item()}')
    return (accuracy*100).item()

class utils:
    create_model = create_model
    create_deep_model = create_deep_model
    train = train
    calculate_accuracy = calculate_accuracy

### Download Data

In [17]:
Path('data').mkdir(exist_ok=True, parents=True)

owncloud.Client.from_public_link('https://uni-bonn.sciebo.de/s/3Uf2gScrvuTPQhB').get_file('/', f'data/steinmetz_2017-01-08_Muller.nc')

True

Extract data needed to train a classifier that can identify stimulus type (left contrast, right contrast, equal contrast) from spikes

In [18]:
dset = xr.load_dataset('data/steinmetz_2017-01-08_Muller.nc')
dset

<xarray.Dataset> Size: 124MB
Dimensions:             (trial: 261, time: 250, cell: 1268, sample: 82,
                         waveform_component: 3, probe: 384, brain_area_lfp: 5,
                         spike_id: 1836009)
Coordinates:
  * trial               (trial) int32 1kB 1 2 3 4 5 6 ... 257 258 259 260 261
  * time                (time) float64 2kB 0.01 0.02 0.03 0.04 ... 2.48 2.49 2.5
  * cell                (cell) int32 5kB 1 2 3 4 5 ... 1264 1265 1266 1267 1268
  * waveform_component  (waveform_component) int32 12B 1 2 3
  * probe               (probe) int32 2kB 1 2 3 4 5 6 ... 380 381 382 383 384
  * brain_area_lfp      (brain_area_lfp) <U5 100B 'CA1' 'DG' 'LP' 'PO' 'VISam'
  * spike_id            (spike_id) int32 7MB 1 2 3 4 ... 1836007 1836008 1836009
Dimensions without coordinates: sample
Data variables: (12/31)
    contrast_left       (trial) int8 261B 50 0 100 0 50 0 ... 100 0 100 0 100 0
    contrast_right      (trial) int8 261B 0 50 25 100 50 50 ... 100 50 100 25 25
    gocue               (trial) float64 2kB 0.9828 0.902 1.114 ... nan nan nan
    stim_onset          (trial) float64 2kB 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5
    feedback_type       (trial) float64 2kB 1.0 1.0 1.0 1.0 ... nan nan nan nan
    feedback_time       (trial) float64 2kB 1.272 1.104 1.402 ... nan nan nan
    ...                  ...
    waveform_w          (cell, sample, waveform_component) float32 1MB 0.0 .....
    waveform_u          (cell, waveform_component, probe) float32 6MB 0.0 ......
    lfp                 (brain_area_lfp, trial, time) float64 3MB -27.6 ... 0...
    spike_time          (spike_id) float32 7MB 2.363 2.385 ... 1.651 0.5142
    spike_cell          (spike_id) uint32 7MB 1 1 1 1 1 ... 1268 1268 1268 1268
    spike_trial         (spike_id) uint32 7MB 1 1 2 2 2 ... 205 205 205 213 252
Attributes:
    session_date:  2017-01-08
    mouse:         Muller
    stim_onset:    0.5
    bin_size:      0.01

In [19]:
spike_cols = ['spike_time', 'spike_cell', 'spike_trial']
df_spikes = dset[spike_cols].to_dataframe().reset_index()

trial_ids = np.sort(df_spikes['spike_trial'].unique())
n_cells = dset.sizes['cell']

# features: spike counts per cell for each trial
features_trials = []
# labels: which side had higher contrast (0=left, 1=right, 2=equal)
labels_trials = []

for trial_id in trial_ids:
    # count spikes per cell
    counts = df_spikes[df_spikes['spike_trial'] == trial_id].groupby('spike_cell').size()
    feature_vector = np.zeros(n_cells)
    feature_vector[counts.index - 1] = counts.values
    features_trials.append(feature_vector)
    
    # create labels
    left = dset['contrast_left'].values[trial_id - 1]
    right = dset['contrast_right'].values[trial_id - 1]
    labels_trials.append(0 if left > right else 1 if right > left else 2)

features = torch.tensor(features_trials, dtype=torch.float32)
labels = torch.tensor(labels_trials, dtype=torch.long)


/tmp/ipykernel_103493/4159512993.py:24: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1752540242079/work/torch/csrc/utils/tensor_new.cpp:254.)
  features = torch.tensor(features_trials, dtype=torch.float32)


Split into train and test data

In [22]:
SEED = 2026
generator = torch.Generator().manual_seed(SEED)

train_fraction = 0.7
test_fraction = 1-train_fraction

features_train, features_test = random_split(features, lengths=[train_fraction, test_fraction], generator=generator)
labels_train, labels_test = random_split(labels, lengths=[train_fraction, test_fraction], generator=generator)

features_train, features_test = features_train.dataset[features_train.indices].float(), features_test.dataset[features_test.indices].float()
labels_train, labels_test = labels_train.dataset[labels_train.indices], labels_test.dataset[labels_test.indices]

In [43]:
mean_cifar10 = (0.4914, 0.4822, 0.4465)
std_cifar10 = (0.2023, 0.1994, 0.2010)

# NOTE: Look up how the mean and std values for normalization are calculated exactly. Simply mean and std across all pixels and images in dataset?
train_data = datasets.CIFAR10(
    download=True,
    root="data",
    train=True,
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean_cifar10, std_cifar10)
    ])
)

test_data = datasets.CIFAR10(
    download=True,
    root="data",
    train=False,
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean_cifar10, std_cifar10)
    ])
)

100%|██████████| 170M/170M [01:36<00:00, 1.78MB/s] 


In [62]:
labels_train = torch.tensor(train_data.targets)
labels_train

tensor([6, 9, 9,  ..., 9, 1, 1])

In [63]:
features_train = torch.tensor(train_data.data, dtype=torch.float32).reshape(-1, 32*32*3)
features_test = torch.tensor(test_data.data, dtype=torch.float32).reshape(-1, 32*32*3)
labels_test = torch.tensor(test_data.targets)

print(f"features_train shape: {features_train.shape}")
print(f"features_test shape: {features_test.shape}")

features_train shape: torch.Size([50000, 3072])
features_test shape: torch.Size([10000, 3072])


## Section 1: Effect of Making Networks Wider

In [70]:
nepochs = 50
n_features = 32 * 32 * 3  # CIFAR-10 images flattened
n_classes = 10  # CIFAR-10 has 10 classes

model = utils.create_model(n_features=n_features, n_classes=n_classes, hidden_layer_size=100)

# create optimizer
optimizer = torch.optim.RMSprop(model.parameters(), lr = 0.01)

# train model
for epoch in tqdm(range(nepochs)):
    utils.train(model, features_train, labels_train, optimizer=optimizer)

utils.calculate_accuracy(model, features_test, labels_test)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:27<00:00,  1.83it/s]

Accuracy: 10.010000228881836


10.010000228881836

## Section 2: Effect of Making Networks Deeper

In [74]:
labels_test.shape

torch.Size([10000])

In [72]:
nepochs = 10
n_features = 32 * 32 * 3
n_classes = 10

model = utils.create_deep_model(n_features=n_features, n_classes=n_classes)

# create optimizer
optimizer = torch.optim.RMSprop(model.parameters(), lr = 0.01)

# train model
for epoch in tqdm(range(nepochs)):
    utils.train(model, features_train, labels_train, optimizer=optimizer)

utils.calculate_accuracy(model, features_test, labels_test)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [01:11<00:00,  7.15s/it]


Accuracy: 10.0


10.0

## Section 3: 